# Cross-Architecture Energy Prediction — Data Processing Pipeline

This notebook loads and processes traces from two sources:
- **Lotaru** (7 machines, runtime + memory only, no energy)
- **Augur** (4 gpgnodes, runtime + memory + RAPL energy)

and builds task-level energy attributions from raw RAPL package/DRAM counters.

## 1. Setup

In [2]:
import pandas as pd
import numpy as np
import os

## 2. Configuration

`machine_folder_map` maps a clean internal machine name (used consistently throughout the pipeline) to the file-naming suffix used inside that machine's own CSV filenames in the Lotaru repo — these differ (e.g. `a1` → `asok01`).

In [3]:
LOTARU_BASE_URL = "https://raw.githubusercontent.com/CRC-FONDA/Lotaru-traces/master/traces"

MACHINE_FOLDER_MAP = {
    "a1":          "asok01",
    "a2":          "asok02",
    "c2":          "c2",
    "local":       "local",
    "n1":          "n1",
    "n2":          "n2",
}

WORKFLOWS = ["atacseq", "bacass", "chipseq", "eager", "methylseq"]

AUGUR_BASE_URL = "../augur/experiments"


## 3. Lotaru Data Loading

**Fixed bug:** `source_workflow` was previously being set to the machine's file-naming suffix (`folder_name`, e.g. `"asok01"`) instead of the actual workflow name. It now correctly stores `workflow`.

In [4]:
def load_lotaru():
    frames = []
    for machine_name, folder_name in MACHINE_FOLDER_MAP.items():
        for workflow in WORKFLOWS:
            url = f"{LOTARU_BASE_URL}/{machine_name}/results_{workflow}/execution_report_{folder_name}.csv"
            try:
                df = pd.read_csv(url)
            except Exception as e:
                print(f"SKIP {machine_name}/{workflow}: {e}")
                continue
            df["source_machine"] = machine_name
            df["source_workflow"] = workflow   
            frames.append(df)

    return pd.concat(frames, ignore_index=True)


In [5]:
lotaru_df = load_lotaru()
print(lotaru_df["source_machine"].value_counts())
print(lotaru_df["source_workflow"].value_counts())

source_machine
a1       1501
a2       1501
c2       1501
n2       1501
n1       1487
local    1333
Name: count, dtype: int64
source_workflow
chipseq      2975
atacseq      2114
eager        1976
methylseq    1099
bacass        660
Name: count, dtype: int64


In [7]:
print(f"Lotaru DF columns ({len(lotaru_df.columns)}):")
print(lotaru_df.columns.tolist())
print(f"{lotaru_df["Workflow"].unique()}")
lotaru_df.head()

Lotaru DF columns (22):
['Label', 'Machine', 'Workflow', 'NumberSequences', 'Task', 'WorkflowInputSize', 'Realtime', '%cpu', 'rss', 'rchar', 'wchar', 'cpus', 'read_bytes', 'write_bytes', 'vmem', 'memory', 'peak_rss', 'TaskInputSize', 'TaskInputSizeUncompressed', 'WorkflowInputUncompressed', 'source_machine', 'source_workflow']
<StringArray>
['atacseq', 'bacass', 'chipseq', 'eager', 'methylseq']
Length: 5, dtype: str


,Label,Machine,Workflow,NumberSequences,Task,WorkflowInputSize,Realtime,%cpu,rss,rchar,...,read_bytes,write_bytes,vmem,memory,peak_rss,TaskInputSize,TaskInputSizeUncompressed,WorkflowInputUncompressed,source_machine,source_workflow
0,train-1,asok01,atacseq,37596126,FASTQC,1862380804,347192,103.2,1067270144,1893391029,...,20480,4407296,4551680000,17179869184,1067270144,1862380804,7045849256,7045849256,a1,atacseq
1,train-1,asok01,atacseq,587600,FASTQC,29105620,12854,176.8,227483648,59997930,...,0,3977216,4551680000,17179869184,227483648,29105620,110119968,110119968,a1,atacseq
2,train-1,asok01,atacseq,4699600,FASTQC,233161053,51047,117.7,759746560,264075396,...,0,4030464,4475781120,17179869184,759746560,233161053,880745968,880745968,a1,atacseq
3,train-1,asok01,atacseq,147000,FASTQC,7287465,8869,204.1,277942272,38172539,...,0,3952640,4555874304,17179869184,277942272,7287465,27547568,27547568,a1,atacseq
4,train-1,asok01,atacseq,18798126,FASTQC,931822514,178182,106.3,1058963456,962817711,...,0,4169728,4549582848,17179869184,1060605952,931822514,3522936056,3522936056,a1,atacseq


## 4. Augur Data Loading — Task Traces

Loads a single run's `trace.csv` (task-level execution data: hostname, timestamps, runtime, memory, I/O). Currently scoped to one workflow/run — see **Section 9 (Next Steps)** for scaling this to all workflows and all 3 replicate runs.

In [ ]:
def load_augur_trace(workflow, cluster,run_id, base_url=AUGUR_BASE_URL):
    url = f"{base_url}/{cluster}/{workflow}/{run_id}/trace.csv"
    df = pd.read_csv(url)
    df["source_workflow"] = workflow
    df["source_run"] = run_id
    return df

augur_chipseq_traces = load_augur_trace("chipseq", "gu-cluster","1")
print(augur_chipseq_traces.shape)
augur_chipseq_traces.head()

(314, 41)


,task_id,hostname,native_id,process,tag,name,status,exit,module,container,...,read_bytes,write_bytes,vol_ctxt,inv_ctxt,workdir,scratch,error_action,cpu_model,source_workflow,source_run
0,2,gpgnode-13,91195,NFCORE_CHIPSEQ:CHIPSEQ:INPUT_CHECK:SAMPLESHEET...,chipseq-input.csv,NFCORE_CHIPSEQ:CHIPSEQ:INPUT_CHECK:SAMPLESHEET...,COMPLETED,0,-,quay.io/biocontainers/python:3.8.3,...,3981312,8192,26,3,/workspace/work/72/149bd7abc51a95175c212b24abdb21,-,-,Intel(R) Xeon(R) CPU E5-2640 v2 @ 2.00GHz,chipseq,1
1,1,gpgnode-14,nf-b695279bfca7c89fb5e1930ade21b687-65ca8,NFCORE_CHIPSEQ:CHIPSEQ:PREPARE_GENOME:CUSTOM_G...,genome.fa,NFCORE_CHIPSEQ:CHIPSEQ:PREPARE_GENOME:CUSTOM_G...,COMPLETED,0,-,quay.io/biocontainers/samtools:1.15.1--h1170115_0,...,5128192,16384,51,2027,/workspace/work/b6/95279bfca7c89fb5e1930ade21b687,-,-,Intel(R) Xeon(R) CPU E5-2640 v2 @ 2.00GHz,chipseq,1
2,3,gpgnode-18,nf-579ef6a7485e229e4add142514104d26-5ad7e,NFCORE_CHIPSEQ:CHIPSEQ:PREPARE_GENOME:GTF2BED,genes.gtf,NFCORE_CHIPSEQ:CHIPSEQ:PREPARE_GENOME:GTF2BED ...,COMPLETED,0,-,quay.io/biocontainers/perl:5.26.2,...,917504,25702400,44,4027,/workspace/work/57/9ef6a7485e229e4add142514104d26,-,-,Intel(R) Xeon(R) CPU E5-2640 v2 @ 2.00GHz,chipseq,1
3,5,gpgnode-16,nf-e9b613123a51bae72b96a36343757761-74075,NFCORE_CHIPSEQ:CHIPSEQ:FASTQC_TRIMGALORE:FASTQC,H3K4me1_LAPC4_Control_REP1_T1,NFCORE_CHIPSEQ:CHIPSEQ:FASTQC_TRIMGALORE:FASTQ...,COMPLETED,0,-,quay.io/biocontainers/trim-galore:0.6.10--hdfd...,...,5677056,2400256,39,13,/workspace/work/e9/b613123a51bae72b96a36343757761,-,-,Intel(R) Xeon(R) CPU E5-2640 v2 @ 2.00GHz,chipseq,1
4,7,gpgnode-14,nf-2eb513a77f4509f6a441a9148c8763b1-663a3,NFCORE_CHIPSEQ:CHIPSEQ:FASTQC_TRIMGALORE:FASTQC,H3K4me1_LAPC4_Control_REP2_T1,NFCORE_CHIPSEQ:CHIPSEQ:FASTQC_TRIMGALORE:FASTQ...,COMPLETED,0,-,quay.io/biocontainers/trim-galore:0.6.10--hdfd...,...,352256,2347008,25,14,/workspace/work/2e/b513a77f4509f6a441a9148c8763b1,-,-,Intel(R) Xeon(R) CPU E5-2640 v2 @ 2.00GHz,chipseq,1


## 5. Energy Processing — Raw RAPL Counters

RAPL package (`pkg.csv`) and DRAM (`dram.csv`) files are **cumulative energy counters** (monotonically increasing), sampled roughly once per second, with a few ms of drift between the two loggers.

**⚠️ Unit conversion not yet verified.** The divisor below (`ENERGY_UNIT_DIVISOR`) was reverse-engineered to produce physically plausible power draw (~140-150W combined pkg+dram for this dual-socket Xeon), but the *actual* unit the logger writes in (true RAPL μJ vs. some other scale) hasn't been confirmed from the logging tool's source/docs. **Confirm this before trusting energy_j at scale** — see Section 9.

In [ ]:
ENERGY_UNIT_DIVISOR = 1_000_000  # verified against the actual RAPL logging tool's units
RAPL_CONSTANTS = {
    "gu-cluster": {"pkg": 65532610987,  "dram": 65532610987},
    "hu-cluster": {"pkg": 262143328850, "dram": 65712999613},
}

PLAUSIBLE_MAX_WATTS = {
    "gu-cluster": 300,
    "hu-cluster": 1000,
}
def load_energy_data(path, max_uj, max_watts):
    df = pd.read_csv(path)
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Detect energy columns: gu-cluster has energy_1 + energy_2 (dual socket),
    # hu-cluster has a single 'energy' column (single socket).
    energy_cols = [c for c in df.columns if c.startswith("energy")]

    total_delta = 0
    for col in energy_cols:
        delta = df[col].diff()
        wrapped = delta < 0
        delta.loc[wrapped] = delta.loc[wrapped] + max_uj
        total_delta = total_delta + delta

    df["interval_j"] = total_delta / ENERGY_UNIT_DIVISOR
    df["time_gap_s"] = df["timestamp"].diff() / 1000
    df["implied_watts"] = df["interval_j"] / df["time_gap_s"]

    suspect = df["implied_watts"] > max_watts
    df.loc[suspect, "interval_j"] = None
    return df


In [ ]:
def load_node_energy(node, run, run_dir, cluster, verbose=True):
    consts = RAPL_CONSTANTS[cluster]
    max_watts = PLAUSIBLE_MAX_WATTS[cluster]

    pkg = load_energy_data(
        os.path.join(run_dir, "energy", f"run_{run}_{node}_pkg.csv"),
        max_uj=consts["pkg"], max_watts=max_watts,
    )

    # DRAM may be empty or missing on some nodes — fall back to pkg-only if so
    dram_path = os.path.join(run_dir, "energy", f"run_{run}_{node}_dram.csv")
    try:
        dram = load_energy_data(dram_path, max_uj=consts["dram"], max_watts=max_watts)
        if dram.empty or dram["interval_j"].notna().sum() == 0:
            raise ValueError("empty dram")
        merged = pd.merge_asof(
            pkg[["timestamp", "interval_j"]],
            dram[["timestamp", "interval_j"]],
            on="timestamp", direction="nearest", tolerance=50,
            suffixes=("_pkg", "_dram"),
        )
        merged["total_interval_j"] = merged["interval_j_pkg"] + merged["interval_j_dram"]
    except (ValueError, pd.errors.EmptyDataError, FileNotFoundError):
        if verbose:
            print(f"  {node}: DRAM missing/empty — using package energy only")
        merged = pkg[["timestamp", "interval_j"]].rename(columns={"interval_j": "total_interval_j"})

    return merged[["timestamp", "total_interval_j"]]

In [ ]:
run_dir = f"{AUGUR_BASE_URL}/gu-cluster/chipseq/1"
nodes = augur_chipseq_traces["hostname"].str.replace("-", "").unique().tolist()
print("Nodes in this run:", nodes)

energy_by_host = {node: load_node_energy(node, 1,run_dir,"gu-cluster") for node in nodes}

Nodes in this run: ['gpgnode13', 'gpgnode14', 'gpgnode18', 'gpgnode16']


## 6. Task-Level Energy Attribution

For each task, sums the node's energy readings that fall inside the task's `[start, complete]` window.

**⚠️ Known limitation — concurrency:** this gives *total node energy while the task ran*, not energy caused *only* by that task. If other tasks ran concurrently on the same node (common in real Nextflow runs), their energy is included too. Confirmed empirically: a 95s task returned 64,066.84 J (~674W implied) — far above what's physically plausible for one task alone on this hardware (~250-300W ceiling), consistent with several overlapping tasks. See Section 7 for the overlap check, and Section 9 for attribution options.

**Also fixed:** inconsistent return values — previously returned the string `"Host Not present"` in one branch and `None` in another. Now consistently returns `None` for both "no data" cases, so downstream code (e.g. `.isna()`) works uniformly.

In [ ]:
def task_energy(task_row, energy_by_host):
    hostname = task_row["hostname"]
    # Find the energy key that matches this hostname
    # (energy keys like 'c40' or 'gpgnode13'; hostnames like 'hu-worker-c40' or 'gpgnode-13')
    host = None
    for key in energy_by_host:
        if key.replace("-", "") in hostname.replace("-", ""):
            host = key
            break
    if host is None:
        return None

    host_energy = energy_by_host[host]
    window = host_energy[
        (host_energy["timestamp"] >= task_row["start"]) &
        (host_energy["timestamp"] <= task_row["complete"])
    ]
    if window.empty:
        return None
    return round(window["total_interval_j"].sum(), 4)

## 7. Validation

Two known test cases:
- **Short task** (259ms, well under the ~1s sampling interval) → correctly returns `None`.
- **Long task** (95s) → returns a value, but flagged above as likely inflated by concurrent tasks on the same node. The overlap check below confirms this.

In [ ]:
short_task = augur_chipseq_traces.iloc[0]
print(short_task[["hostname", "realtime", "start", "complete"]])
print("Energy result:", task_energy(short_task, energy_by_host))

hostname       gpgnode-13
realtime              259
start       1769592537217
complete    1769592537542
Name: 0, dtype: object
Energy result: None


In [ ]:
long_task = augur_chipseq_traces.iloc[1]
print(long_task[["hostname", "realtime", "start", "complete"]])
print("Energy result:", task_energy(long_task, energy_by_host))

hostname       gpgnode-14
realtime            95000
start       1769592538000
complete    1769592633000
Name: 1, dtype: object
Energy result: 5388.2046


In [ ]:
# Check for concurrent tasks on the same node during the long task's window
overlapping = augur_chipseq_traces[
    (augur_chipseq_traces["hostname"] == long_task["hostname"]) &
    (augur_chipseq_traces["start"] < long_task["complete"]) &
    (augur_chipseq_traces["complete"] > long_task["start"]) &
    (augur_chipseq_traces["task_id"] != long_task["task_id"])
]
print(len(overlapping), "overlapping tasks on", long_task["hostname"])
overlapping[["process", "start", "complete"]]

1 overlapping tasks on gpgnode-14


,process,start,complete
4,NFCORE_CHIPSEQ:CHIPSEQ:FASTQC_TRIMGALORE:FASTQC,1769592539000,1769592750000


## 8. Data Quality Summary

In [ ]:
print("Lotaru:")
print(f"  rows: {len(lotaru_df)}, machines: {lotaru_df['source_machine'].nunique()}, "
      f"workflows: {lotaru_df['source_workflow'].nunique()}")

print("\nAugur (chipseq/1 only so far):")
print(f"  rows: {len(augur_chipseq_traces)}, Machines: {augur_chipseq_traces['hostname'].nunique()}")

Lotaru:
  rows: 8824, machines: 6, workflows: 5

Augur (chipseq/1 only so far):
  rows: 314, Machines: 4


# Solving Concurreny issue

In [ ]:
# Check for concurrent tasks on the same node during the long task's window
def check_concurrent_tasks(task_row,trace):
    overlapping= trace[
        (trace["hostname"] == task_row["hostname"]) &
        (trace["start"] < task_row["complete"]) &
        (trace["complete"] > task_row["start"]) &
        (trace["task_id"] != task_row["task_id"])
    ]
    
    return len(overlapping)

In [ ]:
augur_chipseq_traces["concurrent_task_count"] = augur_chipseq_traces.apply(
    lambda row: check_concurrent_tasks(row, augur_chipseq_traces), axis=1
)

In [ ]:
print(augur_chipseq_traces.head())

   task_id    hostname                                  native_id  \
0        2  gpgnode-13                                      91195   
1        1  gpgnode-14  nf-b695279bfca7c89fb5e1930ade21b687-65ca8   
2        3  gpgnode-18  nf-579ef6a7485e229e4add142514104d26-5ad7e   
3        5  gpgnode-16  nf-e9b613123a51bae72b96a36343757761-74075   
4        7  gpgnode-14  nf-2eb513a77f4509f6a441a9148c8763b1-663a3   

                                             process  \
0  NFCORE_CHIPSEQ:CHIPSEQ:INPUT_CHECK:SAMPLESHEET...   
1  NFCORE_CHIPSEQ:CHIPSEQ:PREPARE_GENOME:CUSTOM_G...   
2      NFCORE_CHIPSEQ:CHIPSEQ:PREPARE_GENOME:GTF2BED   
3    NFCORE_CHIPSEQ:CHIPSEQ:FASTQC_TRIMGALORE:FASTQC   
4    NFCORE_CHIPSEQ:CHIPSEQ:FASTQC_TRIMGALORE:FASTQC   

                             tag  \
0              chipseq-input.csv   
1                      genome.fa   
2                      genes.gtf   
3  H3K4me1_LAPC4_Control_REP1_T1   
4  H3K4me1_LAPC4_Control_REP2_T1   

                               

In [ ]:
AUGUR_WORKFLOWS = ["atacseq", "chipseq", "nanoseq", "rnaseq"]
AUGUR_CLUSTERS = ["gu-cluster","hu-cluster"]
AUGUR_RUNS = ["1", "2", "3"]

def get_nodes_for_run(run_dir):
    energy_dir = os.path.join(run_dir, "energy")
    pkg_files = [f for f in os.listdir(energy_dir) if f.endswith("_pkg.csv")]
    return sorted(f.replace("_pkg.csv", "").split("_", 2)[2] for f in pkg_files)

def process_augur_run(workflow, run,cluster):
    run_dir = os.path.join(AUGUR_BASE_URL, cluster,workflow, run)

    # Load + filter trace
    trace = pd.read_csv(os.path.join(run_dir, "trace.csv"))
    trace = trace[trace["status"] == "COMPLETED"].copy()
    trace["start"] = trace["start"].astype(int)
    trace["complete"] = trace["complete"].astype(int)

    # Build energy_by_host for nodes that actually have energy files
    nodes = get_nodes_for_run(run_dir)
    energy_by_host = {}
    for node in nodes:
        try:
            energy_by_host[node] = load_node_energy(node, run,run_dir, cluster,verbose=False)
        except Exception as e:
            print(f"  energy load failed {workflow}/{run}/{node}: {e}")

    # Per-task energy + concurrency (concurrency scoped to THIS run only)
    trace["task_energy_j"] = trace.apply(lambda r: task_energy(r, energy_by_host), axis=1)
    trace["concurrent_task_count"] = trace.apply(lambda r: check_concurrent_tasks(r, trace), axis=1)
    trace["start"]= trace["start"].astype(int)
    trace["complete"]= trace["complete"].astype(int)
    trace["source_workflow"] = workflow
    trace["source_run"] = run
    trace["source_cluster"] = cluster
    for col in ["realtime", "peak_rss", "rchar", "cpus"]:
        trace[col] = pd.to_numeric(trace[col], errors="coerce")

    return trace

# Main loop
all_augur = []
for cluster in AUGUR_CLUSTERS:
    for wf in AUGUR_WORKFLOWS:
        for run in AUGUR_RUNS:
            try:
                df = process_augur_run(wf, run,cluster=cluster)
                all_augur.append(df)
                print(f"OK {wf}/{run}: {len(df)} tasks, {df['task_energy_j'].notna().sum()} with energy")
            except Exception as e:
                print(f"SKIP {wf}/{run}: {e}")

augur_all = pd.concat(all_augur, ignore_index=True)
print(f"\nTotal: {len(augur_all)} tasks across {augur_all['source_workflow'].nunique()} workflows")


OK atacseq/1: 268 tasks, 246 with energy
OK atacseq/2: 268 tasks, 260 with energy
OK atacseq/3: 268 tasks, 248 with energy
OK chipseq/1: 314 tasks, 281 with energy
OK chipseq/2: 314 tasks, 289 with energy
OK chipseq/3: 314 tasks, 295 with energy
OK nanoseq/1: 92 tasks, 89 with energy
OK nanoseq/2: 92 tasks, 86 with energy
OK nanoseq/3: 92 tasks, 91 with energy
OK rnaseq/1: 231 tasks, 228 with energy
OK rnaseq/2: 231 tasks, 225 with energy
OK rnaseq/3: 231 tasks, 222 with energy
OK atacseq/1: 268 tasks, 241 with energy
OK atacseq/2: 268 tasks, 243 with energy
OK atacseq/3: 268 tasks, 239 with energy
OK chipseq/1: 3538 tasks, 3157 with energy
OK chipseq/2: 3538 tasks, 3179 with energy
OK chipseq/3: 3538 tasks, 3152 with energy
OK nanoseq/1: 92 tasks, 86 with energy
OK nanoseq/2: 92 tasks, 87 with energy
OK nanoseq/3: 92 tasks, 86 with energy
OK rnaseq/1: 1269 tasks, 1183 with energy
OK rnaseq/2: 1269 tasks, 1187 with energy
OK rnaseq/3: 1269 tasks, 1178 with energy

Total: 18216 tasks ac

In [ ]:
# What hostnames are in hu's trace?
trace = pd.read_csv("../augur/experiments/hu-cluster/atacseq/1/trace.csv")
print("Trace hostnames:", trace["hostname"].unique())

# What keys are in the energy dict?
run_dir = "../augur/experiments/hu-cluster/atacseq/1"
print("Energy node keys:", get_nodes_for_run(run_dir))

Trace hostnames: <StringArray>
['hu-worker-c45', 'hu-worker-c44', 'hu-worker-c40', 'hu-worker-c42']
Length: 4, dtype: str
Energy node keys: ['c40', 'c42', 'c44', 'c45']


In [ ]:
print(f"Total: {len(augur_all)} tasks")
print(augur_all.groupby("source_cluster")["source_workflow"].value_counts())  # if you added source_cluster

Total: 18216 tasks
source_cluster  source_workflow
gu-cluster      chipseq              942
                atacseq              804
                rnaseq               693
                nanoseq              276
hu-cluster      chipseq            10614
                rnaseq              3807
                atacseq              804
                nanoseq              276
Name: count, dtype: int64


In [ ]:
# If you added a pkg_only flag; if not, add one in load_node_energy and re-run
print(augur_all.groupby("source_cluster")["energy_is_pkg_only"].value_counts())

KeyError: 'Column not found: energy_is_pkg_only'

In [ ]:
print(augur_all.groupby("source_cluster")["task_energy_j"].describe())

                  count          mean           std      min        25%  \
source_cluster                                                            
gu-cluster       2560.0  18810.041256  37882.950243   0.0000  1558.4675   
hu-cluster      14018.0  18276.957842  30849.557890  46.7687  1616.4967   

                       50%           75%          max  
source_cluster                                         
gu-cluster      6927.38900  19753.824925  471888.7077  
hu-cluster      6725.41935  23045.838850  752041.8269  


In [ ]:
augur_agg = (
    augur_all.groupby(["source_workflow", "process", "tag", "hostname"])
    .agg(
        runtime_s=("realtime", "median"),
        peak_mem=("peak_rss", "median"),
        energy_j=("task_energy_j", "median"),
        runtime_std=("realtime", "std"),
        energy_std=("task_energy_j", "std"),
        rchar=("rchar", "median"),
        cpus=("cpus", "median"),
        concurrent_task_count=("concurrent_task_count", "median"),
        n_replicates=("realtime", "count"),
    )
    .reset_index()
)
print(augur_agg.shape)
print(augur_agg["n_replicates"].value_counts().sort_index())
print(augur_agg["hostname"].value_counts())

(13848, 13)
n_replicates
1    10005
2     3352
3      480
4        5
5        2
7        2
8        1
9        1
Name: count, dtype: int64
hostname
hu-worker-c44    3091
hu-worker-c45    3039
hu-worker-c40    2927
hu-worker-c42    2798
gpgnode-14        562
gpgnode-16        478
gpgnode-13        406
gpgnode-18        390
gpgnode-15        157
Name: count, dtype: int64


In [ ]:
augur_agg[augur_agg["n_replicates"] > 3][["source_workflow", "process", "tag", "hostname", "n_replicates"]]

,source_workflow,process,tag,hostname,n_replicates
10207,nanoseq,NFCORE_NANOSEQ:NANOSEQ:BEDTOOLS_UCSC_BIGBED:BE...,-,gpgnode-14,4
10208,nanoseq,NFCORE_NANOSEQ:NANOSEQ:BEDTOOLS_UCSC_BIGBED:BE...,-,gpgnode-16,9
10209,nanoseq,NFCORE_NANOSEQ:NANOSEQ:BEDTOOLS_UCSC_BIGBED:BE...,-,gpgnode-18,4
10210,nanoseq,NFCORE_NANOSEQ:NANOSEQ:BEDTOOLS_UCSC_BIGBED:BE...,-,hu-worker-c40,4
10212,nanoseq,NFCORE_NANOSEQ:NANOSEQ:BEDTOOLS_UCSC_BIGBED:BE...,-,hu-worker-c44,5
10213,nanoseq,NFCORE_NANOSEQ:NANOSEQ:BEDTOOLS_UCSC_BIGBED:BE...,-,hu-worker-c45,7
10318,nanoseq,NFCORE_NANOSEQ:NANOSEQ:PREPARE_GENOME:SAMTOOLS...,genome.fa,gpgnode-13,5
10321,nanoseq,NFCORE_NANOSEQ:NANOSEQ:PREPARE_GENOME:SAMTOOLS...,genome.fa,gpgnode-18,7
10322,nanoseq,NFCORE_NANOSEQ:NANOSEQ:PREPARE_GENOME:SAMTOOLS...,genome.fa,hu-worker-c40,4
10324,nanoseq,NFCORE_NANOSEQ:NANOSEQ:PREPARE_GENOME:SAMTOOLS...,genome.fa,hu-worker-c44,8


In [ ]:
lotaru_spec_data = {
    "node" : ["local","a1","a2","n1","n2","c2"],
    "cores": [8,8,8,8,8,8],
    "ram":[16,32,32,16,16,32],
    "cpu_benchmark":[458,223,223,369,468,523],
    "io_read":[437,306,341,481,481,481],
    "io_write":[415,301,336,483,483,483],
    "bench_time":[36,40,39,33,30,28]
}
lotaru_spec_table = pd.DataFrame(lotaru_spec_data)

In [ ]:
lotaru_spec_table

,node,cores,ram,cpu_benchmark,io_read,io_write,bench_time
0,local,8,16,458,437,415,36
1,a1,8,32,223,306,301,40
2,a2,8,32,223,341,336,39
3,n1,8,16,369,481,483,33
4,n2,8,16,468,481,483,30
5,c2,8,32,523,481,483,28


In [ ]:
AUGUR_GU_SPEC_DATA_URL = "../augur/tool/infrastructure-profiler/profiles"
gu_nodes_id = [13,14,15,16,18]
data =[]

for node in gu_nodes_id:
    df = pd.read_csv(f"{AUGUR_GU_SPEC_DATA_URL}/gpgnode-{node}.csv")
    df = df.drop("z7b",axis=1)
    df = df.rename(columns={"sysbench":"cpu_benchmark"})
    df_reordered = df.loc[:,["node","cores","ram","cpu_benchmark","io_read","io_write","bench_time"]]
    data.append(df_reordered)

augur_gu_spec_table = pd.concat(data, ignore_index=True)
augur_gu_spec_table
    

,node,cores,ram,cpu_benchmark,io_read,io_write,bench_time
0,gpgnode-13,32,64,305.89,97.6,90.6,275.251000
1,gpgnode-14,32,64,306.01,109.0,90.9,230.926333
2,gpgnode-15,32,64,305.84,109.0,90.8,182.168000
3,gpgnode-16,32,64,306.15,109.0,91.4,197.361000
4,gpgnode-18,32,64,306.29,109.0,91.7,261.806000


In [ ]:
AUGUR_HU_SPEC_URL= "../augur/tool/infrastructure-profiler/profiles"
hu_nodes = ["c40","c42","c44","c45"]
hu_data=[]
for node in hu_nodes:
    df = pd.read_csv(f"{AUGUR_HU_SPEC_URL}/hu-{node}.csv")
    df = df.drop("z7b",axis=1)
    df = df.rename(columns={"sysbench":"cpu_benchmark"})
    df_reordered = df.loc[:,["node","cores","ram","cpu_benchmark","io_read","io_write","bench_time"]]
    hu_data.append(df_reordered)

augur_hu_spec_table = pd.concat(hu_data,ignore_index=True)

augur_hu_spec_table

,node,cores,ram,cpu_benchmark,io_read,io_write,bench_time
0,hu-c40,32,256,1056.95,1362,952,23.160667
1,hu-c42,32,256,1049.80,1380,972,24.036333
2,hu-c44,32,256,1070.40,421,492,33.091667
3,hu-c45,32,256,1066.66,420,499,34.571667


In [ ]:
print(f"Lotaru Specs:\n {lotaru_spec_table}")
print(f"GU Specs:\n {augur_gu_spec_table}")
print(f"HU Specs:\n {augur_hu_spec_table}")

Lotaru Specs:
     node  cores  ram  cpu_benchmark  io_read  io_write  bench_time  \
0  local      8   16            458      437       415          36   
1     a1      8   32            223      306       301          40   
2     a2      8   32            223      341       336          39   
3     n1      8   16            369      481       483          33   
4     n2      8   16            468      481       483          30   
5     c2      8   32            523      481       483          28   

  source_dataset  
0         lotaru  
1         lotaru  
2         lotaru  
3         lotaru  
4         lotaru  
5         lotaru  
GU Specs:
          node  cores  ram  cpu_benchmark  io_read  io_write  bench_time
0  gpgnode-13     32   64         305.89     97.6      90.6  275.251000
1  gpgnode-14     32   64         306.01    109.0      90.9  230.926333
2  gpgnode-15     32   64         305.84    109.0      90.8  182.168000
3  gpgnode-16     32   64         306.15    109.0      91.4  1

In [ ]:
lotaru_spec_table["source_dataset"] = "lotaru"
augur_gu_spec_table["source_dataset"]= "gu_cluster"
augur_hu_spec_table["source_dataset"] = "hu_cluster"
combined_spec_table = pd.concat([lotaru_spec_table,augur_gu_spec_table,augur_hu_spec_table], ignore_index=True)

In [ ]:
combined_spec_table

,node,cores,ram,cpu_benchmark,io_read,io_write,bench_time,source_dataset
0,local,8,16,458.00,437.0,415.0,36.000000,lotaru
1,a1,8,32,223.00,306.0,301.0,40.000000,lotaru
2,a2,8,32,223.00,341.0,336.0,39.000000,lotaru
3,n1,8,16,369.00,481.0,483.0,33.000000,lotaru
4,n2,8,16,468.00,481.0,483.0,30.000000,lotaru
5,c2,8,32,523.00,481.0,483.0,28.000000,lotaru
6,gpgnode-13,32,64,305.89,97.6,90.6,275.251000,gu_cluster
7,gpgnode-14,32,64,306.01,109.0,90.9,230.926333,gu_cluster
8,gpgnode-15,32,64,305.84,109.0,90.8,182.168000,gu_cluster
9,gpgnode-16,32,64,306.15,109.0,91.4,197.361000,gu_cluster
